# GP2 Cohort Harmonization Pipeline

This notebook turns a cohort's raw data files into one harmonized dataset, driven by a **coding sheet** (a Google Sheet) that says, per output variable: which raw file and column it comes from, how to transform it, and what values are valid.

The notebook runs in four parts:

1. **Setup** -> Install packages, get the helper code, connect to Google Sheets.
2. **Sync the coding sheet** -> make sure your cohort's coding sheet matches the latest GP2 reference data dictionary.
3. **Prepare the data environment** -> mount Drive, point the notebook at your raw files.
4. **Process the coding sheet** -> the actual harmonization run: lint the sheet, process every item, merge everything into one dataset, and save the output.

> For a step-by-step walkthrough with troubleshooting tips, see the accompanying SOP document.

## Part 0: Setup

Run these cells once per Colab session: install the `pygsheets` package, get this repo's helper code (`code/`) onto the runtime, and connect to Google Sheets.

In [ ]:
# Install pygsheets
!pip install --upgrade -q pygsheets

In [ ]:
# --- Colab setup: clone the repo so code/ is available locally ---
# Skip/edit this cell if you're already running inside a cloned copy of the
# repo (e.g. running locally, or you mounted/cloned it another way).
import os

REPO_URL = 'https://github.com/<your-org>/<your-repo>.git'  # <- set this
REPO_DIR = '<your-repo>'  # <- local folder name git clone will create

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())


In [ ]:
# Install packages
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import glob
import sys

# Make the local `code` package importable. `code/` must be a
# subfolder of the current working directory (true if you ran the clone
# cell above, or if you're running this notebook from the repo root
# locally/in Drive).
#
# NOTE: Python's standard library has its own top-level module named
# `code` (code.py, an interactive-console helper). sys.path.append()
# would put the repo root AFTER the stdlib paths, so the stdlib `code`
# would shadow this repo's `code/` package. Using insert(0, ...) instead
# puts the repo root FIRST, so the local package wins.
REPO_ROOT = os.getcwd()
if REPO_ROOT in sys.path:
    sys.path.remove(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

# If Python already imported the stdlib `code` module earlier
# in this session (e.g. this cell was re-run), drop it from the module
# cache so the next import below re-resolves to the local package.
if 'code' in sys.modules and not hasattr(sys.modules['code'], '__path__'):
    del sys.modules['code']


### Connect to Google Sheets

Signs you in interactively (Colab will prompt for account access) and gives you a client (`gc`) used throughout the rest of the notebook to open and edit Google Sheets.

In [ ]:
# Read the coding sheet
from code.sheets_io import get_pygsheets_client, ReplaceSheet

gc = get_pygsheets_client()


## Part 1: Sync Your Coding Sheet With the GP2 Data Dictionary

The GP2 Data Dictionary is the master list of variables every cohort's coding sheet should cover. This section:

- Creates your cohort's coding sheet from the reference dictionary if it doesn't exist yet, or
- Compares your existing coding sheet against the *latest* reference and flags any differences, so new/changed items don't get missed.

### 1.1 Get the latest reference dictionary

In [ ]:
# Load the reference data dictionary
!git clone https://github.com/GP2code/GP2-Data-Dictionary.git

In [ ]:
ref_dictionary_path = 'GP2-Data-Dictionary/GP2_Data_Dictionary_ver1.1-3.csv'  # kept as a variable for the run manifest below
ref = pd.read_csv(ref_dictionary_path)
print(ref.shape)

### 1.2 Open (or create) your cohort's coding sheet

Paste your cohort's Google Sheet key below. If the sheet's `Sheet1` tab is empty, it will be initialized from the reference dictionary automatically.

In [ ]:
# Cohort coding -> add the coding sheet's key
sheet = gc.open_by_key('') # copy Google sheet key and paste here
worksheet = sheet.worksheet_by_title('Sheet1')
d = worksheet.get_as_df(include_tailing_empty=False)
print(d.shape)
if d.shape[0]==0: # empty then copy ref
  dnew = ref[['Single Measure', 'Item', 'Description', 'ItemType', 'Required', 'Values']].copy()
  dnew[['File', 'KeyList', 'Operation', 'Action']]=np.nan
  # Update sheet 1
  worksheet = sheet.worksheet_by_title('Sheet1')
  worksheet.set_dataframe(dnew, start='A1',  fit=True, nan='')
  print('"Sheet1" CREATED from sheet_ref')
  d = dnew.copy()

### 1.3 Compare your coding sheet to the reference dictionary

In [ ]:
k1 = d[['Item', 'Description', 'ItemType', 'Required', 'Values']].astype(str).agg('-'.join, axis=1)
k2 = ref[['Item', 'Description', 'ItemType', 'Required', 'Values']].astype(str).agg('-'.join, axis=1)
not_in_d = ref.loc[~k2.isin(k1), ['Single Measure', 'Item', 'Description', 'ItemType', 'Required', 'Values']]
not_in_ref = d.loc[~k1.isin(k2), ['Item', 'Description', 'ItemType', 'Required', 'Values']]
print(not_in_d.shape)
print(not_in_ref.shape)

Save any differences found to a `Dic_conflict` tab on the coding sheet, for review:

In [ ]:
d_conflict = pd.merge(not_in_d, not_in_ref, on='Item', suffixes=['_ref', ''], how='outer')

# Save the dictionary conflict
new_sheet = 'Dic_conflict'
ReplaceSheet(sheet, new_sheet, d_conflict)

### 1.4 If conflicts were found, resolve them

Open the `Dic_conflict` tab and review each row:

- Items that only appear in the **reference** dictionary (`_ref` columns) are new or changed upstream: decide whether your cohort should have them, and whether they can be derived from your available raw data.
- Items that only appear in **your** coding sheet may need to be reconciled with the reference (renamed, removed, or confirmed as cohort-specific).

Once resolved, the cell below writes a merged `updated_Sheet1` tab: replace your sheet's `Sheet1` tab with it once you're satisfied.

In [ ]:
# Update the Sheet1
d2 = pd.merge(ref[['Single Measure', 'Item', 'Description', 'ItemType', 'Required', 'Values']],
              d[['Item', 'File', 'KeyList', 'Operation', 'Action']],
              on='Item', how='left')

# Save the updated dictionary
ReplaceSheet(sheet, 'updated_Sheet1', d2)

## Part 2: Prepare the Data Environment

Mount Google Drive and point the notebook at your cohort's raw data files.

### 2.0 Mount Google Drive

In [ ]:
# Mount gdrive
from google.colab import drive
drive.mount("/content/drive")

### 2.1 Set your cohort name and copy raw data into the working folder

Update `cohort`, `datafolder`, and `workfolder` for your cohort before running this cell.

In [ ]:
# cd to the work folder and copy the raw data to the processing data
cohort = 'cohort'
datafolder = f'/content/drive/dir/raw_data/{cohort}'
workfolder = f'/content/drive/dir/processing/{cohort}'

os.chdir(workfolder)
!cp -r "{datafolder}"/data_combined.csv . # if possible, directly accessing the data is a cleaner use of the workfolder

### 2.2 Load QC helper functions

In [ ]:
# Useful functions -> now defined in code/qc_utils.py
from code.qc_utils import checkDup, checkNull, TakeOneEntry

### 2.3 Confirm the raw files are in place

In [ ]:
!ls {workfolder}

## Part 3: Pre-Processing (Optional)

Use this section for any one-off data cleaning, transformation, or derivation that needs to happen *before* the coding sheet is processed (e.g. combining files, fixing known data-entry errors). Leave it empty if nothing is needed for this cohort.

## Part 4: Process the Coding Sheet

This is the main harmonization run. A few reminders about the coding sheet before running it:

- **Numeric/integer items** need a range condition in the `Values` column, e.g. `(y>=0) & (y<=40)`.
- **String items** list their allowed values in the `Values` column, e.g. `['M', 'F']`.
- Delete any columns from the coding sheet that this script doesn't use. Extra columns can confuse the read-in step.

### 4.1 Define pre-cleaning fixes (optional)

`cleaning_list` holds row-level fixes (remove or change specific raw values) applied to raw files before per-item processing. Leave it empty (`[]`) if none are needed.

In [ ]:
cleaning_list = []

### 4.2 Load the current coding sheet

In [ ]:
worksheet = sheet.worksheet_by_title('Sheet1')
d = worksheet.get_as_df(empty_value=np.nan)

### 4.3 Pre-flight check: lint the coding sheet

Catch mistakes in the coding sheet *before* spending a run processing files, instead of discovering them one `try`/`except`-caught error at a time. This only checks the sheet's own syntax/consistency. 
> **Fix anything printed here before proceeding.**

In [ ]:
from code.coding_sheet_lint import lint_coding_sheet, print_lint_results

lint_problems = lint_coding_sheet(d)
print_lint_results(lint_problems)

### 4.4 Run the main processing step

For every raw file referenced in the coding sheet: loads the file, applies any pre-cleaning fixes, resolves duplicate rows per KeyList, applies each item's operation (`rename`/`map`/`der1`), and runs required/value-range checks.

In [ ]:
from code.processing import process_coding_sheet

a, processing_errors, d = process_coding_sheet(d, cleaning_list=cleaning_list)

### 4.5 Review processing errors

Items that failed to process (bad column name, malformed `map`/`der1` expression, unrecognized operation, etc.) are collected here instead of silently halting the run. Check this before trusting the merged output below. Anything listed here did **not** make it into the processed data and will be missing from the final file until the coding sheet is fixed and this notebook is re-run.

In [ ]:
processing_errors_df = pd.DataFrame(processing_errors)
print(f"{len(processing_errors_df)} item(s) failed to process")
processing_errors_df

### 4.6 Merge all KeyList groups into one dataset

Joins every KeyList group together: longitudinal items (keyed by `participant_id` + `visit_month`) form the base grain, and static/baseline items (keyed by `participant_id` alone) are broadcast across every visit row.

In [ ]:
from code.merge_keylists import merge_keylist_groups

x, b = merge_keylist_groups(a)

### 4.7 Save the merged output

In [ ]:
# from datetime import datetime

# today = datetime.today().strftime("%Y-%m-%d")

In [ ]:
# NOTE: this cell was missing from the original notebook -> output_path is
# referenced in the manifest step below but was never defined. Added here
# so the merged data is actually saved and the manifest step has a real path.
output_path = f'/content/drive/dir/processed/{cohort}'/{cohort}_processed.csv'
x.to_csv(output_path, index=False)
print(f"Saved merged output to {output_path} (shape={x.shape})")

### 4.8 Record a run manifest and log the run

Writes a small JSON manifest recording what ran, when, against which coding sheet, and how many errors occurred — useful provenance for explaining exactly how a given output file was produced. Also appends a summary row to the coding sheet's `run_log` tab.

In [ ]:
from code.manifest import build_manifest, save_manifest, log_run_to_sheet

manifest = build_manifest(
    cohort=cohort,
    sheet=sheet,
    ref_dictionary_path=ref_dictionary_path,
    ref=ref,
    d=d,
    processing_errors=processing_errors,
    output_path=output_path,
    output_shape=x.shape,
)

manifest_path = save_manifest(manifest, output_path)
log_run_to_sheet(sheet, manifest, new_sheet='run_log')
